# Unpack

In [1]:
from pyunpack import Archive

Archive('train.7z').extractall("")

ValueError: archive file does not exist:/home/emastr/github/aipek_task_2026/notebooks/train.7z

# Extract features

We extract the features in the same format as the nnUnet framework, for the sake of setting a standard, And possibly to enable the usage of a pretrained nnUnet model.


In [2]:
import json
import os
import shutil

patient_dirs_raw = set(os.listdir("../data_raw/train/raw_data"))
patient_dirs_der = set(os.listdir("../data_raw/train/derivatives"))
patient_dirs_phe = set(os.listdir("../data_raw/train/phenotype"))
print(f"All patient directories match: {(patient_dirs_raw == patient_dirs_der) and (patient_dirs_raw == patient_dirs_phe)}")


hold_out = "sub-stroke0001,sub-stroke0011,sub-stroke0022,sub-stroke0040,sub-stroke0057,sub-stroke0077,sub-stroke0087,sub-stroke0097,sub-stroke0107,sub-stroke0117,sub-stroke0140,sub-stroke0150,sub-stroke0161,sub-stroke0171,sub-stroke0181".split(",")
print(f"Hold out patients: {hold_out}")

root_img_tr = "../data/Dataset/imagesTr"
root_lab_tr = "../data/Dataset/labelsTr"
root_img_ts = "../data/Dataset/imagesTs"
root_lab_ts = "../data/Dataset/labelsTs"

os.makedirs(root_img_tr, exist_ok=True)
os.makedirs(root_lab_tr, exist_ok=True)
os.makedirs(root_img_ts, exist_ok=True)
os.makedirs(root_lab_ts, exist_ok=True)

for patient_dir in patient_dirs_raw:
    root_img = root_img_tr if patient_dir not in hold_out else root_img_ts
    root_lab = root_lab_tr if patient_dir not in hold_out else root_lab_ts
    patient_id = patient_dir[-4:]
    shutil.copyfile(f"../data_raw/train/raw_data/{patient_dir}/ses-01/{patient_dir}_ses-01_ncct.nii.gz",
                    f"{root_img}/case_{patient_id}_0000.nii.gz")
    shutil.copyfile(f"../data_raw/train/derivatives/{patient_dir}/ses-01/{patient_dir}_ses-01_space-ncct_cta.nii.gz",
                    f"{root_lab}/case_{patient_id}_0001.nii.gz")
    
    
dataset_json = {
    "channel_names": {
        "0": "NCCT",
        "1": "CTA"
    },
    "numTraining": len(patient_dirs_raw) - len(hold_out),
    "numTest": len(hold_out),
    "file_ending": ".nii.gz",
}

with open("../data/Dataset/dataset.json", "w") as f:
    f.write(json.dumps(dataset_json, indent=4))

All patient directories match: True
Hold out patients: ['sub-stroke0001', 'sub-stroke0011', 'sub-stroke0022', 'sub-stroke0040', 'sub-stroke0057', 'sub-stroke0077', 'sub-stroke0087', 'sub-stroke0097', 'sub-stroke0107', 'sub-stroke0117', 'sub-stroke0140', 'sub-stroke0150', 'sub-stroke0161', 'sub-stroke0171', 'sub-stroke0181']
